In [4]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time
from pathlib import Path

C:\Users\Demerchant_Hart\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
BANK_APPS = {
    'Monzo'    : 'co.uk.getmondo',
    'Starling' : 'com.starlingbank.android',
    'Barclays' : 'com.barclays.android.barclaysmobilebanking',
    'HSBC'     : 'uk.co.hsbc.hsbcukmobilebanking',
    'NatWest'  : 'com.rbs.mobile.android.natwest',
    'Lloyds'   : 'com.grppl.android.shell.CMBlloydsTSB73',
}

OUTPUT_PATH = Path('../data/raw/reviews_raw.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

all_dfs = []

for bank_name, app_id in BANK_APPS.items():
    print(f'Scraping {bank_name}...')
    bank_reviews = []
    continuation_token = None

    for batch in range(25):  # 25 batches x 200 = 5,000 reviews per bank
        try:
            result, continuation_token = reviews(
                app_id,
                lang='en',
                country='gb',
                sort=Sort.NEWEST,
                count=200,
                continuation_token=continuation_token,
            )
            bank_reviews.extend(result)

            if not continuation_token or not result:
                print(f'  No more reviews at batch {batch+1}')
                break

            if (batch + 1) % 5 == 0:
                print(f'  Batch {batch+1}: {len(bank_reviews):,} reviews so far')

            time.sleep(0.5)

        except Exception as e:
            print(f'  Error at batch {batch+1}: {e}')
            break

    # Convert to DataFrame
    df = pd.DataFrame(bank_reviews)
    df['bank'] = bank_name
    df['app_id'] = app_id
    all_dfs.append(df)
    print(f'  Done: {len(df):,} reviews')

    # Save after each bank
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(OUTPUT_PATH, index=False)
    print(f'  Progress saved.')
    time.sleep(2)

# Final summary
df_raw = pd.read_csv(OUTPUT_PATH)
print()
print('='*50)
print('SCRAPING COMPLETE')
print('='*50)
print(f'Total reviews : {len(df_raw):,}')
print()
print('Reviews per bank:')
print(df_raw['bank'].value_counts().to_string())
print()
print('Rating distribution:')
print(df_raw['score'].value_counts().sort_index().to_string())
print()
print('Columns:', list(df_raw.columns))

Scraping Monzo...
  Batch 5: 1,000 reviews so far
  Batch 10: 2,000 reviews so far
  Batch 15: 3,000 reviews so far
  Batch 20: 4,000 reviews so far
  Batch 25: 5,000 reviews so far
  Done: 5,000 reviews
  Progress saved.
Scraping Starling...
  Batch 5: 1,000 reviews so far
  Batch 10: 2,000 reviews so far
  Batch 15: 3,000 reviews so far
  Batch 20: 4,000 reviews so far
  Batch 25: 5,000 reviews so far
  Done: 5,000 reviews
  Progress saved.
Scraping Barclays...
  Batch 5: 1,000 reviews so far
  Batch 10: 2,000 reviews so far
  Batch 15: 3,000 reviews so far
  Batch 20: 4,000 reviews so far
  Batch 25: 5,000 reviews so far
  Done: 5,000 reviews
  Progress saved.
Scraping HSBC...
  Batch 5: 1,000 reviews so far
  Batch 10: 2,000 reviews so far
  Batch 15: 3,000 reviews so far
  Batch 20: 4,000 reviews so far
  Batch 25: 5,000 reviews so far
  Done: 5,000 reviews
  Progress saved.
Scraping NatWest...
  Batch 5: 1,000 reviews so far
  Batch 10: 2,000 reviews so far
  Batch 15: 3,000 revi

In [6]:
import pandas as pd

df = pd.read_csv('../data/raw/reviews_raw.csv', parse_dates=['at'])

print("Date range:", df['at'].min(), "→", df['at'].max())
print()
print("Sample 1-star review (NatWest):")
sample = df[(df['bank']=='NatWest') & (df['score']==1)].iloc[0]
print(f"  Date   : {sample['at']}")
print(f"  Content: {sample['content'][:300]}")
print()
print("Sample 5-star review (Monzo):")
sample2 = df[(df['bank']=='Monzo') & (df['score']==5)].iloc[0]
print(f"  Date   : {sample2['at']}")
print(f"  Content: {sample2['content'][:300]}")
print()
print("Average review length (chars):")
df['review_len'] = df['content'].str.len()
print(df.groupby('bank')['review_len'].mean().round(0).sort_values(ascending=False).to_string())
print()
print("Null review text:", df['content'].isnull().sum())

Date range: 2024-06-13 04:50:28 → 2026-06-02 13:59:42

Sample 1-star review (NatWest):
  Date   : 2026-06-02 12:53:15
  Content: locked out the app, pain in the arse to get back in!

Sample 5-star review (Monzo):
  Date   : 2026-06-02 09:45:48
  Content: amazing app

Average review length (chars):
bank
HSBC        104.0
Monzo        90.0
NatWest      87.0
Starling     70.0
Barclays     62.0
Lloyds       51.0

Null review text: 1


In [7]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,bank,app_id,review_len
0,de2e33fa-8b9f-4d25-b850-ccde652be126,Áine,https://play-lh.googleusercontent.com/a-/ALV-U...,amazing app,5,0,7.27.0,2026-06-02 09:45:48,NaN,NaN,7.27.0,Monzo,co.uk.getmondo,11.0
1,8b63c112-adc9-4102-a10d-c5177e111cb2,john ganz,https://play-lh.googleusercontent.com/a-/ALV-U...,love monzo it's so easy to use few people said...,5,0,7.27.0,2026-06-02 08:25:55,NaN,NaN,7.27.0,Monzo,co.uk.getmondo,79.0
2,d658f4cd-ed2d-40a4-92be-5f86d3ab9555,shane harte,https://play-lh.googleusercontent.com/a-/ALV-U...,easy to use the app doesn't seem to crash like...,4,0,7.28.0,2026-06-01 20:49:59,NaN,NaN,7.28.0,Monzo,co.uk.getmondo,71.0
3,2c687841-9cd7-408b-94b6-6ba94f1ea374,jamie mcleod,https://play-lh.googleusercontent.com/a-/ALV-U...,amazing,5,0,7.28.0,2026-06-01 18:01:44,NaN,NaN,7.28.0,Monzo,co.uk.getmondo,7.0
4,177a4332-2d6b-459e-97a1-4d8e21aa31ab,Joshua Marsh,https://play-lh.googleusercontent.com/a/ACg8oc...,amazing bank and app,5,0,7.28.0,2026-06-01 16:40:14,NaN,NaN,7.28.0,Monzo,co.uk.getmondo,20.0
